# 08 — Passing-lane geometric eligibility and ambiguity audit

**Purpose:** Distinguish (i) no defender projected on the **open** pass segment, (ii) lack of opponent coordinates, and (iii) carrier/teammate missingness. Compute separate **endpoint proximity** for opponents, rather than mislabeling the absence of an interior projection as a missing geometric measurement. Retain this as descriptive geometry, not pass feasibility or pressing.

This notebook uses the verified Notebook 05 canonical files and Notebook 07 candidate files in `outputs/`. It does not load continuous tracking or pose. It does not use realized pass outcomes. No thresholds are tuned.

**Output to upload:** `outputs/P2_lane_semantics_gate.json`. Full diagnostic rows stay local.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime,timezone
import json, math, statistics
ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None: raise FileNotFoundError('Run within the SkillCorner repository.')
OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
CANON=sorted(OUT.glob('P1_canonical_decisions_match_*_ROLES.jsonl'))
assert len(CANON)==20, f'Expected 20 canonical matches, found {len(CANON)}'
print('Found',len(CANON),'matches')

Found 20 matches


## Define geometry independently with synthetic unit tests

Interior projections have `0 < t < 1`; endpoints are tracked separately. A defender beyond either endpoint is not counted as an interior obstruction. For every candidate we also record the nearest defender to the **full closed segment**, which includes endpoints. Missing opponents are explicitly distinguished from zero interior projections.

In [2]:
def valid(x):
    return (isinstance(x,(list,tuple)) and len(x)==2 and
            all(isinstance(v,(float,int)) and math.isfinite(v) for v in x))
def dist(a,b):return math.hypot(a[0]-b[0],a[1]-b[1])
def segment_projection(a,b,p):
    dx,dy=b[0]-a[0],b[1]-a[1];sq=dx*dx+dy*dy
    if sq <= 1e-12:return None
    t=((p[0]-a[0])*dx+(p[1]-a[1])*dy)/sq
    clipped=max(0.,min(1.,t))
    q=[a[0]+clipped*dx,a[1]+clipped*dy]
    return t,dist(p,q)
def actors(r,team):
    return [(str(p['player_id']),p['xy'],quality)
            for quality in ('detected','extrapolated')
            for p in r.get(f'{team}_{quality}',[]) if valid(p.get('xy'))]

assert segment_projection([0,0],[10,0],[5,2])==(0.5,2.0)
assert segment_projection([0,0],[10,0],[-2,0])==(-0.2,2.0)
assert segment_projection([0,0],[10,0],[12,0])==(1.2,2.0)
assert segment_projection([0,0],[0,0],[1,1]) is None
print('Synthetic projection tests passed')

Synthetic projection tests passed


## Reconcile every candidate with its original player positions

The validation checks the per-match keys and candidate counts. Every valid-carrier decision is expected to generate one candidate per other teammate with finite coordinates, but we do **not** force the total to be ten: the observed counts are verified rather than presumed. Candidate records with an interior projection are compared against Notebook 07 up to numerical tolerance.

In [3]:
match_reports=[]; totals=Counter(); discrepancy_examples=[]
for file in CANON:
    mid=file.name.split('_match_')[1].split('_ROLES')[0]
    candidate_file=OUT/f'P2_pass_lane_candidates_match_{mid}.jsonl'
    if not candidate_file.is_file():raise FileNotFoundError(candidate_file)
    decisions={}; lines=0
    with file.open() as stream:
        for line in stream:
            r=json.loads(line);key=(int(r['period']),int(r['frame']),str(r['carrier_player_id']))
            if key in decisions:raise AssertionError(f'{mid} duplicate decision key {key}')
            decisions[key]=r;lines+=1
    matched_by_decision=Counter();count=Counter();samples=[]
    path=OUT/f'P2_lane_semantics_match_{mid}.jsonl'
    with candidate_file.open() as stream,path.open('w') as dest:
        for line in stream:
            v=json.loads(line);count['candidate_pairs']+=1
            key=(int(v['period']),int(v['frame']),str(v['carrier_player_id']))
            if key not in decisions:raise AssertionError(f'Unmatched candidate: {mid}, {key}')
            r=decisions[key];matched_by_decision[key]+=1
            carrier=r.get('carrier_xy');pid=str(v['teammate_player_id'])
            tm={p:str(quality) for p,xy,quality in actors(r,'teammates')}
            if pid not in tm:raise AssertionError(f'Unmatched teammate: {mid} {key} {pid}')
            # Preserve player coordinates from canonical source; check candidate distance.
            target=next(xy for p,xy,q in actors(r,'teammates') if p==pid)
            if not valid(carrier):raise AssertionError(f'Candidate has missing carrier: {mid} {key}')
            if abs(dist(carrier,target)-float(v['pass_segment_length_m']))>1e-7:
                raise AssertionError(f'Candidate distance mismatch: {mid} {key} {pid}')
            opp=actors(r,'opponents')
            projections=[(p,t,d,q) for p,xy,q in opp
                         if (projection:=segment_projection(carrier,target,xy)) is not None
                         for t,d in [projection]]
            interior=[(p,t,d,q) for p,t,d,q in projections if 0<t<1]
            if not opp:status='missing_opponents'
            elif dist(carrier,target)<1e-6:status='degenerate_segment'
            elif interior:status='interior_projection_present'
            else:status='no_interior_projection'
            count[status]+=1
            measured=min((d for p,t,d,q in interior),default=None)
            published=v.get('interior_opponent_clearance_m')
            consistent=(measured is None and published is None) or (measured is not None and published is not None and abs(measured-published)<1e-7)
            if not consistent:
                count['clearance_disagreement']+=1
                if len(samples)<5:samples.append({'period':key[0],'frame':key[1],'teammate':pid,'published':published,'recomputed':measured})
            endpoint=min((d for p,t,d,q in projections if t<=0 or t>=1),default=None)
            record={'match_id':mid,'period':key[0],'frame':key[1],'carrier_player_id':key[2],
                'teammate_player_id':pid,'geometry_status':status,
                'interior_opponent_clearance_m':measured,
                'nearest_endpoint_defender_distance_m':endpoint,
                'nearest_full_segment_defender_distance_m':min((d for p,t,d,q in projections),default=None),
                'n_finite_opponents':len(opp),'n_interior_projecting_opponents':len(interior),
                'carrier_detected':r.get('carrier_detected') is True,
                'teammate_quality':tm[pid]}
            dest.write(json.dumps(record,separators=(',',':'))+'\n')
    for key,r in decisions.items():
        valid_tm=len(actors(r,'teammates')) if valid(r.get('carrier_xy')) else 0
        if matched_by_decision[key]!=valid_tm:
            raise AssertionError(f'Candidate count mismatch {mid} {key}: {matched_by_decision[key]} vs {valid_tm}')
    count['decisions']=lines;count['decisions_with_candidates']=len(matched_by_decision)
    match_reports.append({'match_id':mid,**dict(count),'disagreement_examples':samples})
    totals.update(count)
    print(mid, 'pairs:',count['candidate_pairs'], 'no interior projection:',count['no_interior_projection'])
assert totals['clearance_disagreement']==0, f'Clearance mismatch {totals["clearance_disagreement"]}'
print('Geometry cross-check completed')

1874553 pairs: 9560 no interior projection: 221
1886347 pairs: 9990 no interior projection: 256
1899585 pairs: 9290 no interior projection: 187
1925299 pairs: 10620 no interior projection: 289
1927964 pairs: 11390 no interior projection: 306
1953632 pairs: 9700 no interior projection: 185
1959846 pairs: 9450 no interior projection: 193
1986691 pairs: 9150 no interior projection: 181
1996435 pairs: 9970 no interior projection: 175
1996436 pairs: 7850 no interior projection: 190
2006229 pairs: 9770 no interior projection: 173
2006363 pairs: 9760 no interior projection: 219
2007448 pairs: 8620 no interior projection: 222
2007721 pairs: 9590 no interior projection: 263
2010085 pairs: 9350 no interior projection: 209
2011166 pairs: 7550 no interior projection: 165
2013725 pairs: 9170 no interior projection: 220
2015213 pairs: 8810 no interior projection: 183
2016236 pairs: 9590 no interior projection: 199
2017461 pairs: 8390 no interior projection: 210
Geometry cross-check completed


In [4]:
report={'created_utc':datetime.now(timezone.utc).isoformat(),
 'objective':'Distinguish absence of interior defender projection from missing position data; independently verify Notebook 07 geometric calculations',
 'matches':len(match_reports),'totals':dict(totals),'per_match':match_reports,
 'interpretation':[
  'no_interior_projection means no measured defender projects between carrier and target at the decision frame; it does not establish a clear or feasible pass',
  'interior_projection_present does not automatically mean an interception is possible',
  'endpoint proximity is reported separately from open-segment clearance',
  'same-frame snapshots only; no direction normalization, defender movement, ball travel or outcome-derived predictors',
  'missing opponent positions and degenerate segments have distinct explicit states',
  'full coordinate and candidate-level diagnostics remain local']}
dest=OUT/'P2_lane_semantics_gate.json'
dest.write_text(json.dumps(report,indent=2))
print(json.dumps(report['totals'],indent=2));print('UPLOAD:',dest)

{
  "candidate_pairs": 187570,
  "interior_projection_present": 183324,
  "no_interior_projection": 4246,
  "decisions": 18792,
  "decisions_with_candidates": 18757
}
UPLOAD: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P2_lane_semantics_gate.json
